# Stage 2 — MATLAB ↔ Python GPU rho parity

Bu aşama mevcut environment'taki ikinci-order/Gauss-Hermite korelasyon bloklarını taşır:

\[
\texttt{compute\_ch\_rho\_avg}
\]

ve

\[
\texttt{compute\_ch\_eff\_rho\_avg\_fast}.
\]

Kontrol edilen çıktılar:

\[
\rho_{RB},\rho_{BR},\rho_{RU},\rho_{UR},\rho_{\text{RU-hop}}.
\]

Önce `float64/complex128` parity, sonra `float32/complex64` production error ölçülür.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import sys
import pandas as pd
import torch

ROOT = Path('/content/drive/MyDrive/MyDrive/RIS')
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))

MODULE = ROOT / 'ris_gpu_rho_stage2.py'
if not MODULE.exists():
    MODULE = Path('/content/ris_gpu_rho_stage2.py')

assert MODULE.exists(), f"Modül bulunamadı: {MODULE}"
sys.path.insert(0, str(MODULE.parent))

from ris_gpu_rho_stage2 import compare_rho_matlab_case

print("Loaded:", MODULE)

## MATLAB golden case

MATLAB'da mevcut environment değişkenlerin hazırken:

```matlab
export_rho_parity_case( ...
    "rho_case.mat", ...
    gnb2ris,ris2ue,geometry,c0,lambda_0, ...
    K_BR,K_RU,lsp_BR,lsp_RU);
```

Sonra `rho_case.mat` dosyasını Colab `/content` altına yükle.

In [ ]:
CASE = Path('/content/rho_case.mat')
assert CASE.exists(), "rho_case.mat dosyasını /content altına yükle."
print(CASE)

In [ ]:
# DOUBLE PARITY
m = compare_rho_matlab_case(
    str(CASE),
    device='cuda' if torch.cuda.is_available() else 'cpu',
    parity=True,
    gh_pair_chunk=80,
)

parity_df = pd.DataFrame([m])
display(parity_df.T.rename(columns={0:'value'}))

# İkinci-order GH hesaplarında eig/GH ve reduction ordering nedeniyle
# Stage-1 kadar 1e-15 beklemiyoruz. İlk kabul eşiği 1e-10.
rel_cols = [c for c in parity_df.columns if c.endswith('_relFro')]
worst = parity_df[rel_cols].to_numpy().max()

print("Worst relative Frobenius error:", worst)
assert worst < 1e-10, (
    f"Stage-2 parity henüz geçmedi. worst={worst:.3e}"
)

print("PASS: Stage-2 MATLAB ↔ Python rho parity")

In [ ]:
# PRODUCTION FLOAT32 / COMPLEX64
m32 = compare_rho_matlab_case(
    str(CASE),
    device='cuda' if torch.cuda.is_available() else 'cpu',
    parity=False,
    gh_pair_chunk=80,
)

prod_df = pd.DataFrame([m32])
display(prod_df.T.rename(columns={0:'value'}))

print(
    "Bu değerler production GPU precision'ın MATLAB-double "
    "rho referansına göre hatasıdır."
)

## Sonraki aşama

Bu parity geçince Stage 3'e geçeceğiz:

\[
UBR,\quad URU,\quad \mu_{\rm Feff},\quad \sigma^2_{\rm Feff},
\quad C,\quad \mu_{\rm SNR},\quad \sigma^2_{\rm Wick}.
\]

Burada rho'ları bank içinde bir kere üretip yüzlerce RIS candidate için yeniden kullanacağız.
Asıl candidate-batch GPU hızlanması Stage 3'ten itibaren daha kritik hale gelecek.